In [1]:
import pandas as pd

csv= pd.read_csv('/media/Volume/data/MIMIC_IV/discharge.csv.gz')
print(csv.head())
# print unique subject IDs
unique_subject_ids = csv['subject_id'].unique()
print("Unique subject IDs in discharge.csv:", len(unique_subject_ids))

# total len
print("Total number of ecgs:", len(csv))

          note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   
1  10000032-DS-22    10000032  22841357        DS        22   
2  10000032-DS-23    10000032  29079034        DS        23   
3  10000032-DS-24    10000032  25742920        DS        24   
4  10000084-DS-17    10000084  23052089        DS        17   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   
1  2180-06-27 00:00:00  2180-07-01 10:15:00   
2  2180-07-25 00:00:00  2180-07-25 21:42:00   
3  2180-08-07 00:00:00  2180-08-10 05:43:00   
4  2160-11-25 00:00:00  2160-11-25 15:09:00   

                                                text  
0   \nName:  ___                     Unit No:   _...  
1   \nName:  ___                     Unit No:   _...  
2   \nName:  ___                     Unit No:   _...  
3   \nName:  ___                     Unit No:   _...  
4   \nName:  ___                    Unit No:   __...  
Unique s

In [2]:
# add column for LVEF values
csv['lvef'] = None

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd

# Choose a light model
model_id = "google/gemma-2-9b-it"

# Load tokenizer and model (4-bit quantized)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cuda",
    torch_dtype=torch.float16,
    load_in_4bit=True  # needs bitsandbytes
)
for i, text in enumerate(csv["text"]):
    prompt = f"""Extract only the LVEF value (number or range) from the text, for example:
    ... LVEF>55%   --> >55
    Doppler elt his LVEF was ~30% and reduced ...  --> 30
    HFpEF (LVEF 50% ___, PAD, CKD (stage IV), prior DVT c/b severe --> 50
    Her TTE showed mild-moderate mitral and moderate tricuspid regurgitation, LVEF 50-55%, and pulmonary hypertension. --> 50-55

    if no LVEF value is found, return "unknown".
    text: {text}
    
    Answer: The LVEF value is:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=15)
        print(outputs)

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = decoded.replace(prompt, "").strip()
    print(f"Row {i}: {answer}")
    csv.at[i, 'lvef'] = answer

# for each row in the dataframe, generate LVEF value

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/857 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

/home/luna97/Research/MIT-BIH_ecg_arrhytmia/mit_bih_env/lib/python3.11/site-packages/bitsandbytes/nn/modules.py:457: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(


tensor([[    2, 62088,  1297,  ...,   108,   107,     1]], device='cuda:0')
Row 0: 
tensor([[    2, 62088,  1297,  ...,   107,   108,     1]], device='cuda:0')
Row 1: Not found in the text.
tensor([[    2, 62088,  1297,  ...,   107,   108,     1]], device='cuda:0')
Row 2: Not found in the text.
tensor([[    2, 62088,  1297,  ...,   107,   108,     1]], device='cuda:0')
Row 3: Not available in the text.
tensor([[    2, 62088,  1297,  ...,   110,   107,     1]], device='cuda:0')
Row 4: 
tensor([[    2, 62088,  1297,  ...,   110,   107,     1]], device='cuda:0')
Row 5: 
tensor([[    2, 62088,  1297,  ...,   110,   107,     1]], device='cuda:0')
Row 6: 
tensor([[    2, 62088,  1297,  ...,   110,   107,     1]], device='cuda:0')
Row 7: 
tensor([[    2, 62088,  1297,  ...,   110,   107,     1]], device='cuda:0')
Row 8: 
tensor([[    2, 62088,  1297,  ...,   107,   108,     1]], device='cuda:0')
Row 9: 55
tensor([[     2,  62088,   1297,  ..., 235308,    107,      1]],
       device='cuda:0')

KeyboardInterrupt: 

In [ ]:
# save the dataframe to a new CSV file
csv.to_csv('/media/Volume/data/MIMIC_IV/discharge_lvef.csv')

